In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-in2nhdnd/unsloth_a5dbc5ff3274470baee34bfb75fad024
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-in2nhdnd/unsloth_a5dbc5ff3274470baee34bfb75fad024
  Resolved https://github.com/unslothai/unsloth.git to commit d1e312dcdc57bf020aa0f6da810226efe79cd69a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.6/289.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 75.6 MB/s eta 0:00:00
 

In [2]:
from unsloth import FastLanguageModel
import torch
from huggingface_hub import login

# 1. Log in to Hugging Face
login()

# 2. Configuration
max_seq_length = 2048
dtype = None
load_in_4bit = True

# 3. Load the Model and Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("SUCCESS: Model is loaded!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find Config class in trl.trainer.dpo_trainer. Found: []
Unsloth: Could not find Config class in trl.trainer.iterative_sft_trainer. Found: []
Unsloth: Could not find Config class in trl.trainer.sft_trainer. Found: []


==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

SUCCESS: Model is loaded!


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("SUCCESS: LoRA adapters attached!")

Unsloth 2025.11.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


SUCCESS: LoRA adapters attached!


In [4]:
from datasets import load_dataset

# 1. Define the Instruction Format
alpaca_prompt = """Below is a medical report containing technical jargon. Rewrite it in simple, patient-friendly language.

### Medical Report:
{}

### Simplified Explanation:
{}"""

# 2. Download the Data
dataset = load_dataset("GEM/cochrane-simplification", split="train", revision="refs/convert/parquet")

# 3. Format the Data
def formatting_prompts_func(examples):
    inputs = examples["source"]
    outputs = examples["target"]
    texts = []
    for input, output in zip(inputs, outputs):
        text = alpaca_prompt.format(input, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

print("SUCCESS: Dataset loaded and formatted!")
print(f"Example data point:\n{dataset['text'][0][:200]}...")

default/train/0000.parquet:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/1.05M [00:00<?, ?B/s]

default/test/0000.parquet:   0%|          | 0.00/1.24M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3568 [00:00<?, ? examples/s]

SUCCESS: Dataset loaded and formatted!
Example data point:
Below is a medical report containing technical jargon. Rewrite it in simple, patient-friendly language.

### Medical Report:
Two trials met the inclusion criteria. One compared 2% ketanserin ointment ...


In [5]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

Map (num_proc=2):   0%|          | 0/3568 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,568 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.078200
2,1.913700
3,1.853600
4,1.959200
5,1.955700
6,1.829100
7,1.911700
8,1.921600
9,1.692600
10,1.677900


In [6]:
from google.colab import drive
import os

drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/Medical_Llama_Adapter"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"SUCCESS: Model saved permanently to {save_path}")

Mounted at /content/drive
SUCCESS: Model saved permanently to /content/drive/MyDrive/Medical_Llama_Adapter


In [7]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

medical_report = """
Patient presented with dyspnea on exertion and peripheral edema.
Echocardiogram revealed distinct left ventricular hypertrophy
with reduced ejection fraction (35%).
Diagnosis: Congestive Heart Failure.
"""

prompt = alpaca_prompt.format(
    medical_report,
    ""
)

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)

result = tokenizer.batch_decode(outputs)[0]
simplified_text = result.split("### Simplified Explanation:")[-1].replace("<|end_of_text|>", "")

print("🏥 ORIGINAL REPORT:")
print(medical_report)
print("-" * 30)
print("✨ SIMPLIFIED EXPLANATION:")
print(simplified_text.strip())

🏥 ORIGINAL REPORT:

Patient presented with dyspnea on exertion and peripheral edema.
Echocardiogram revealed distinct left ventricular hypertrophy
with reduced ejection fraction (35%).
Diagnosis: Congestive Heart Failure.

------------------------------
✨ SIMPLIFIED EXPLANATION:
This patient experienced difficulty breathing and swelling in their legs.
The patient's heart was examined using an ultrasound test,
which revealed that the heart was enlarged and not pumping blood effectively.
Based on these findings, the patient was diagnosed with Congestive Heart Failure.
This is a condition where the heart is unable to pump blood effectively,
causing fluid to build up in the lungs and other parts of the body.<|eot_id|>


In [13]:
!pip install streamlit
!npm install localtunnel
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

⠙⠹⠸⠼⠴⠦⠧⠇
up to date, audited 23 packages in 1s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠇Collecting unsloth@ git+https://github.com/unslothai/unsloth.git (from unsloth[colab-new]@ git+https://github.com/unslothai/unsloth.git)
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-172rtp2c/unsloth_adcf113be37349c6969950743e25bf19
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-172rtp2c/unsloth_adcf113be37349c6969950743e25bf19
  Resolved https://github.com/unslothai/unsloth.git to commit d1e312dcdc57bf020aa0f6da810226efe79cd69a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.

  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0


In [9]:
%%writefile app.py
import streamlit as st
from unsloth import FastLanguageModel
import torch
from transformers import TextStreamer

# 1. Page Configuration
st.set_page_config(page_title="Medical Simplifier AI", page_icon="🏥", layout="centered")
st.title("🏥 Medical Report Simplifier")
st.markdown("### Turn complex doctor notes into plain English.")

# 2. Load Model
@st.cache_resource
def load_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "/content/drive/MyDrive/Medical_Llama_Adapter",
        max_seq_length = 2048,
        dtype = None,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)
    return model, tokenizer
with st.spinner("Waking up the AI Doctor... (This takes 30 seconds)"):
    model, tokenizer = load_model()

# 3. Define the Prompt Style
alpaca_prompt = """Below is a medical report containing technical jargon. Rewrite it in simple, patient-friendly language.

### Medical Report:
{}

### Simplified Explanation:
{}"""

# 4. The User Interface
input_text = st.text_area("Paste Medical Report Here:", height=200, placeholder="Patient presents with acute pharyngitis...")

if st.button("Simplify Report ✨"):
    if input_text:
        inputs = tokenizer(
            [alpaca_prompt.format(input_text, "")],
            return_tensors = "pt"
        ).to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens = 512,
            use_cache = True
        )

        decoded = tokenizer.batch_decode(outputs)[0]
        cleaned_text = decoded.split("### Simplified Explanation:")[-1].replace("<|end_of_text|>", "")

        st.success("Here is the simplified version:")
        st.write(cleaned_text)
    else:
        st.warning("Please enter some text first!")

st.markdown("---")
st.caption("Powered by Llama-3 & Unsloth | Student GenAI Project")

Writing app.py


In [ ]:
!wget -q -O - ipv4.icanhazip.com

!streamlit run app.py & npx localtunnel --port 8501

In [12]:
import gradio as gr
from unsloth import FastLanguageModel

# 1. Load the Saved Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/Medical_Llama_Adapter",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# 2. Define the Simplifier Function
def simplify_report(text):
    alpaca_prompt = """Below is a medical report containing technical jargon. Rewrite it in simple, patient-friendly language.

### Medical Report:
{}

### Simplified Explanation:
{}"""

    inputs = tokenizer(
        [alpaca_prompt.format(text, "")],
        return_tensors = "pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 512,
        use_cache = True
    )

    decoded = tokenizer.batch_decode(outputs)[0]
    return decoded.split("### Simplified Explanation:")[-1].replace("<|end_of_text|>", "")

# 3. Create the Interface
demo = gr.Interface(
    fn=simplify_report,
    inputs=gr.Textbox(label="Paste Medical Report Here", lines=6, placeholder="Patient presents with..."),
    outputs=gr.Textbox(label="Simplified Version", lines=6),
    title="🏥 Medical Report Simplifier",
    description="This AI turns complex clinical notes into plain English for patients.",
    examples=[
        ["Patient presents with acute pharyngitis and cervical lymphadenopathy."],
        ["Diagnosis: Congestive Heart Failure with reduced ejection fraction."]
    ]
)

# 4. Launch it!
demo.launch(share=True)

==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cb740508067b82f8a7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
